# example of reading one trace

In [ ]:
using Pkg

cd(@__DIR__)
Pkg.activate("../")
using Dates, SonifSismo, CairoMakie

In [ ]:
archive = DataArchive("/Users/nobuaki/Documents/mtFujiContinuous")

In [ ]:

available_stations(archive)
available_channels(archive)

In [ ]:
start_time = DateTime(2008, 5, 28)  # UTC
end_time   = DateTime(2008, 5, 30, 0, 0, 30)     # UTC

traces = read_window(
    archive,
    start_time,
    end_time;
    stations="FUJ",
    channels=["wE", "wN", "wU"],
    merge=true,
 #    merge_gaps=:zero,       # or :linear
    processed=false,
    #bandpass=(0.5, 15.0),
)

In [ ]:
include("/Users/nobuaki/Documents/Github/sonifSismo.jl/examples/sonify_one_trace.jl")

trace_wU = only(t for t in traces if strip(t.sta.cha) == "wU")


In [ ]:

plot_events = read_catalog(
    archive;
    start=start_time,
    stop=end_time,
)

wavefig = plot_waveforms(
    traces;
    #events=plot_events,
    title="FUJ event window",
)
audio = sonify_trace(
    trace_wU;
    acceleration=128,
    gain=4,
    saturation=:soft,
)
wavefig

In [ ]:

waveform_video(
    traces,
    audio,
    "../audio/FUJ_waveforms.mp4";
    framerate=30,
    window_seconds=60,
    title="FUJ: sonified vertical component",
)

In [ ]:
specfig = plot_time_frequency(
    traces;
    #events=plot_events,
    window=2.0,
    overlap=0.75,
    title="FUJ time-frequency power",
)

In [ ]:

plain = sonify_trace(trace_wU; acceleration=128)
bright = sonify_trace(trace_wU; acceleration=128, harmonics=(0.18, 0.08), octave=3)
higher = sonify_trace(trace_wU; acceleration=128, octave=3)

write_sonification("audio/plain.wav", plain)
write_sonification("audio/bright.wav", bright)
write_sonification("audio/higher.wav", higher)

#wavplay(bright.signal, bright.sample_rate)

In [ ]:

plain = sonify_trace(trace_wU; acceleration=128)

soft = sonify_trace(
    trace_wU;
    acceleration=128,
    gain=4,
    saturation=:soft,
)

hard = sonify_trace(
    trace_wU;
    acceleration=128,
    gain=4,
    saturation=:hard,
)
write_sonification("audio/plain2.wav", plain)
write_sonification("audio/soft.wav", bright)
write_sonification("audio/hard.wav", hard)
#wavplay(plain.signal, plain.sample_rate)
#wavplay(soft.signal, soft.sample_rate)
#wavplay(hard.signal, hard.sample_rate)

In [ ]:
level=0.92          # final maximum amplitude, between 0 and 1
gain=4              # boost before clipping/saturation
saturation=:soft    # smooth tanh saturation
saturation=:hard    # abrupt clipping, more distorted

In [ ]:
# Quieter overall output
quiet = sonify_trace(trace_wU; acceleration=100, level=0.4)

# Slightly louder, gentle saturation
gentle = sonify_trace(trace_wU; acceleration=100, gain=2, saturation=:soft)

# Much louder/denser
dense = sonify_trace(trace_wU; acceleration=100, gain=6, saturation=:soft)

# Strong distorted/transient sound
clipped = sonify_trace(trace_wU; acceleration=100, gain=4, saturation=:hard)

In [ ]:
mwe = run_lfe_mwe()
wavplay(mwe.louder.signal, mwe.louder.sample_rate)